## Environment Setup and Dependency Installation

This section prepares the execution environment in Google Colab and installs all required dependencies needed for the code summarization pipeline.



In [2]:
!pip install transformers==4.37.2
!pip install sentence-transformers==2.6.1
!pip install peft==0.8.2
!pip install accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 32.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

## Mount Google Drive

The notebook uses **Google Drive for persistent storage** of datasets, checkpoints, and generated outputs.

Mounting Google Drive allows the notebook to access both:

1. **Provided assignment resources**
2. **A working directory used to store outputs generated during execution**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

SHARED_DIR = "/content/drive/MyDrive/Assignment-2/"
WORK_DIR = "/content/drive/MyDrive/genai_assignment2_alice/"

os.makedirs(WORK_DIR, exist_ok=True)

print("Shared folder:", SHARED_DIR)
print("Working folder:", WORK_DIR)

Mounted at /content/drive
Shared folder: /content/drive/MyDrive/Assignment-2/
Working folder: /content/drive/MyDrive/genai_assignment2_alice/


## Verifying Access to Assignment Resources

Before continuing, the notebook verifies that the shared folder is accessible and prints its contents.

In [4]:
import os

print("Path:", SHARED_DIR)

if os.path.exists(SHARED_DIR):
    print("Folder exists")
    print("Contents:", os.listdir(SHARED_DIR))
else:
    print("Folder NOT found")

Path: /content/drive/MyDrive/Assignment-2/
Folder exists
Contents: ['dataset', 'sample_code.txt', 'sample_summary.txt', 'requirements.txt', 'get_codet5_embeddings.py']


In [5]:
!pip install torch
!pip install -r {SHARED_DIR}requirements.txt
!pip install gitpython tqdm requests

ERROR: Could not find a version that satisfies the requirement torch==2.0.1 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.0.1


## Repository Mining for Dataset Construction

The assignment requires constructing a dataset of approximately **50,000 Java code–summary pairs mined from public GitHub repositories**. To satisfy this requirement, the notebook automatically searches for and downloads Java repositories from GitHub.

This section sets up the directory where repositories will be stored and defines a function that retrieves suitable repositories using the **GitHub Search API**.

---

### Repository Storage Directory

A dedicated directory is created within the working directory to store cloned repositories:


In [6]:
REPO_DIR = WORK_DIR + "repos/"
os.makedirs(REPO_DIR, exist_ok=True)

### Searching for Java Repositories

The function `search_java_repos()` queries the GitHub API to collect repositories that meet specific criteria.


The function retrieves up to **100 Java repositories** with the following filters:

- `language:Java` → ensures repositories primarily contain Java code  
- `stars:>50` → selects projects with at least 50 stars, which increases the likelihood of finding well-documented code and meaningful comments  

Repositories are sorted by popularity to prioritize widely used projects that are more likely to contain useful Javadoc comments that can serve as natural language summaries.

---

### Handling GitHub API Pagination

GitHub's search API returns results in **pages of up to 30 repositories**. The function iterates through multiple pages until the desired number of repositories is collected.

---
The function returns a list of repository clone URLs:
```
return repos[:max_repos]
```

In [7]:
import requests
import time

def search_java_repos(max_repos=100):

    repos = []
    page = 1

    while len(repos) < max_repos:

        url = "https://api.github.com/search/repositories"

        params = {
            "q": "language:Java stars:>50",
            "sort": "stars",
            "order": "desc",
            "per_page": 30,
            "page": page
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print("GitHub API error:", response.text)
            break

        data = response.json()

        for item in data["items"]:
            repos.append(item["clone_url"])

        print(f"Collected {len(repos)} repos")

        page += 1
        time.sleep(2)  # avoid rate limit

    return repos[:max_repos]

### Selecting Source Repositories

After defining the repository search function, the notebook retrieves a list of candidate repositories that will be used to construct the dataset.


In [8]:
repos = search_java_repos(120)

print("Total repos:", len(repos))
print(repos[:5])

Collected 30 repos
Collected 60 repos
Collected 90 repos
Collected 120 repos
Total repos: 120
['https://github.com/Snailclimb/JavaGuide.git', 'https://github.com/krahets/hello-algo.git', 'https://github.com/GrowingGit/GitHub-Chinese-Top-Charts.git', 'https://github.com/iluwatar/java-design-patterns.git', 'https://github.com/macrozheng/mall.git']


### Cloning Selected Repositories

After collecting the list of candidate repositories, the next step is to **download their source code locally** so that Java files can be scanned for method–summary pairs.

The following function clones each repository into the working directory:

```python
def clone_repo(repo):
  ```

  Before cloning, the code checks whether the repository already exists locally:
  ```
  if os.path.exists(path):
    print("Already cloned:", name)
    return
  ```
  This prevents unnecessary re-downloading if the notebook is rerun. This is important because cloning repositories can be time-consuming and GitHub may impose rate limits.

In [9]:
from git import Repo
from concurrent.futures import ThreadPoolExecutor
import os

def clone_repo(repo):

    name = repo.split("/")[-1].replace(".git","")
    path = REPO_DIR + name

    # This prevents recloning
    if os.path.exists(path):
        print("Already cloned:", name)
        return

    try:
        print("Cloning:", name)
        Repo.clone_from(repo, path, depth=1)
    except Exception as e:
        print("Failed:", repo)


NUM_THREADS = 6

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    executor.map(clone_repo, repos)

Already cloned:Already cloned: mall
Already cloned: java-design-patterns
Already cloned: GitHub-Chinese-Top-Charts
Already cloned: hello-algo
Already cloned: JavaGuide
 spring-boot
Already cloned: elasticsearch
Already cloned: LeetCodeAnimation
Already cloned: ghidra
Already cloned: Java
Already cloned: advanced-java
Already cloned: interviews
Already cloned: spring-framework
Already cloned: jadx
Already cloned: dbeaver
Already cloned: guava
Already cloned: RxJava
Already cloned: termux-app
Already cloned: JeecgBoot
Already cloned: dubbo
Already cloned: NewPipe
Already cloned: tutorials
Already cloned: arthas
Already cloned: hello-algorithm
Already cloned: leetcode
Already cloned: lottie-android
Already cloned: awesome-system-design-resources
Already cloned: glide
Already cloned: netty
Already cloned: selenium
Already cloned: spring-boot-demo
Already cloned: zxing
Already cloned: easyexcel
Already cloned: MPAndroidChart
Already cloned: halo
Already cloned: AndroidUtilCode
Already clone

KeyboardInterrupt: 

Failed: https://github.com/openjdk/jdk.git


### Verifying Repository Download

After cloning the repositories, this step verifies that the repositories were successfully downloaded to the working directory.


In [10]:
import os
print(len(os.listdir(REPO_DIR)), "repositories cloned")

120 repositories cloned


### Scanning Repositories for Java Source Files

After cloning the repositories, the next step is to locate all Java source files that may contain methods and Javadoc comments.

The notebook scans each repository and collects the paths of files ending in `.java`.

```python
for repo in tqdm(repos, desc="Scanning repos"):
```
---
### Limiting Extremely Large Repositories

Some repositories contain tens of thousands of files, which can dramatically slow down the scanning and extraction process. To prevent this, a file limit is applied:
```
MAX_FILES = 5000

file_count = sum(len(files) for _, _, files in os.walk(repo_path))

if file_count > MAX_FILES:
    print(f"Skipping large repo: {repo} ({file_count} files)")
```

In [11]:
import os
from tqdm import tqdm

MAX_FILES = 5000   # skip repos larger than this
java_files = []

repos = os.listdir(REPO_DIR)

for repo in tqdm(repos, desc="Scanning repos"):
    repo_path = os.path.join(REPO_DIR, repo)

    # count files quickly
    file_count = sum(len(files) for _, _, files in os.walk(repo_path))

    if file_count > MAX_FILES:
        print(f"Skipping large repo: {repo} ({file_count} files)")
        continue

    for root, dirs, files in os.walk(repo_path):
        for f in files:
            if f.endswith(".java"):
                java_files.append(os.path.join(root, f))

print("Total Java files found:", len(java_files))

Scanning repos:   1%|          | 1/120 [00:11<23:21, 11.77s/it]


KeyboardInterrupt: 

### Caching the List of Java Files

Scanning all cloned repositories for Java source files can be time-consuming, especially when the notebook is executed multiple times. To avoid repeating this expensive operation, the notebook saves the list of discovered Java files to disk and reloads it when available.

The path used for this cache file is:
```
genai_assignment2_alice/java_files.json
```

If the file already exists, the notebook loads the stored list of Java file paths:
```
if os.path.exists(java_file_path):
    print("Loading saved Java file list...")
```

In [12]:
import os
import json

java_file_path = WORK_DIR + "java_files.json"

if os.path.exists(java_file_path):
    print("Loading saved Java file list...")
    with open(java_file_path) as f:
        java_files = json.load(f)

else:
    print("Scanning repositories for Java files...")

    java_files = []

    for repo in os.listdir(REPO_DIR):
        repo_path = os.path.join(REPO_DIR, repo)

        for root, dirs, files in os.walk(repo_path):
            for f in files:
                if f.endswith(".java"):
                    java_files.append(os.path.join(root, f))

    print("Found", len(java_files), "Java files")

    with open(java_file_path, "w") as f:
        json.dump(java_files, f)

    print("Saved java file list.")

Loading saved Java file list...


After saving the list of discovered Java files to `java_files.json`, this step verifies that the file was written successfully and contains data.


In [13]:
import os

path = os.path.join(WORK_DIR, "java_files.json")
print(os.path.exists(path))
print(os.path.getsize(path))

True
20374583


## Extracting Code–Summary Pairs

This section begins the process of constructing the dataset used to train the code summarization model.


The natural language summaries are obtained from **Javadoc comments**, which typically describe the purpose of a method.

These `(method, summary)` pairs form the dataset used to train the LSTM model.

---

### Checkpoints
Extracting method–summary pairs from thousands of Java files can take a long time. To prevent losing progress if execution is interrupted, the notebook periodically saves intermediate results.
```
pairs_checkpoint = WORK_DIR + "code_summary_pairs_partial.json"
```
### Final Dataset File
Once extraction completes, the full dataset will be written to:
```
genai_assignment2_alice/code_summary_pairs.json
```
This file will contain the final list of (method, summary) pairs used to construct the training and validation datasets.

In [14]:
import os
import json
import re
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

pairs_checkpoint = WORK_DIR + "code_summary_pairs_partial.json"
pairs_final = WORK_DIR + "code_summary_pairs.json"

pattern = re.compile(
    r"/\*\*(.*?)\*/\s*((public|protected|private|static|\s)+[^{]+\{[^}]*\})",
    re.DOTALL
)

MAX_PAIRS = 100000
SAVE_INTERVAL = 2000

pairs = []

if os.path.exists(pairs_checkpoint):
    print("Loading checkpoint...")
    with open(pairs_checkpoint) as f:
        pairs = json.load(f)

print("Starting with", len(pairs), "pairs")

Loading checkpoint...
Starting with 98000 pairs


## Processing Individual Java Files

The function `process_file()` scans a single Java file and extracts valid `(method, summary)` pairs.


In [15]:
def process_file(file):

    results = []

    try:
        with open(file, encoding="utf-8", errors="ignore") as f:
            code = f.read()

        # skip files with no Javadoc
        if "/**" not in code:
            return results

        matches = pattern.findall(code)

        for comment, method, _ in matches:

            summary = comment.strip().split("\n")[0]
            summary = re.sub(r"\*", "", summary).strip()

            method = re.sub(r"\s+", " ", method)

            if len(summary) > 5:
                results.append((method, summary))

    except:
        pass

    return results

### Parallel Dataset Extraction

Processing thousands of Java files sequentially would be slow. To accelerate the process, extraction is parallelized using a thread pool:
```
with ThreadPoolExecutor(max_workers=8) as executor:
```

In [16]:
with ThreadPoolExecutor(max_workers=8) as executor:

    for result in tqdm(executor.map(process_file, java_files), total=len(java_files)):

        pairs.extend(result)

        if len(pairs) >= MAX_PAIRS:
            print("Reached target pair count")
            break

        if len(pairs) % SAVE_INTERVAL == 0 and len(pairs) > 0:

            with open(pairs_checkpoint, "w") as f:
                json.dump(pairs, f)

            print("Checkpoint saved:", len(pairs))

print("Extraction finished. Total pairs:", len(pairs))

  0%|          | 1/123989 [00:00<24:37:34,  1.40it/s]

Checkpoint saved: 98000


  1%|          | 1328/123989 [02:26<3:46:15,  9.04it/s]


Reached target pair count
Extraction finished. Total pairs: 100000


### Final Dataset Output

Once extraction is complete, the full dataset is saved to:
```
genai_assignment2_alice/code_summary_pairs.json
```

This file contains the complete list of extracted (method, summary) pairs and will be used in the next stage to construct the training and validation datasets.

In [17]:
with open(pairs_final, "w") as f:
    json.dump(pairs, f)

print("Final dataset saved:", len(pairs))

Final dataset saved: 100000


### Cleaning and Filtering Extracted Pairs

After extracting `(method, summary)` pairs from Java files, the dataset is further cleaned to remove incorrect or low-quality examples. This improves the quality of the training data and helps the model learn more meaningful relationships between code and summaries.

The cleaning process applies several filters to each extracted pair.

---

### Removing Class Definitions

The regular expression used during extraction may occasionally capture **class declarations** instead of methods.

Example of an unwanted capture:

```java
public class Example { ... }

In [18]:
import re

clean_pairs = []

for method, summary in pairs:

    # remove class definitions accidentally captured
    if " class " in method:
        continue

    # require parentheses (real methods)
    if "(" not in method or ")" not in method:
        continue

    # normalize whitespace
    method = re.sub(r"\s+", " ", method).strip()

    # lowercase summary
    summary = summary.lower().strip()

    # remove short or bad summaries
    if len(summary) < 10:
        continue

    clean_pairs.append((method, summary))

print("Clean pairs:", len(clean_pairs))

Clean pairs: 52933


### Limiting the Dataset Size

After cleaning and filtering the extracted `(method, summary)` pairs, the dataset is capped at a fixed size

In [19]:
MAX_DATASET = 50000

clean_pairs = clean_pairs[:MAX_DATASET]

print("Final dataset size:", len(clean_pairs))

Final dataset size: 50000


### Create Train/Validation Split

In [20]:
train_pairs = clean_pairs[:49000]
val_pairs = clean_pairs[49000:50000]

print("Train size:", len(train_pairs))
print("Validation size:", len(val_pairs))

Train size: 49000
Validation size: 1000


### Define output file paths

In [21]:
train_code_path = WORK_DIR + "train_code.txt"
train_summary_path = WORK_DIR + "train_summary.txt"

val_code_path = WORK_DIR + "val_code.txt"
val_summary_path = WORK_DIR + "val_summary.txt"

### Writing the Training and Validation Dataset Files

After constructing and splitting the cleaned dataset into training and validation sets, the data is written to text files.  

The dataset is formatted as **plain text files with one example per line**, where:

- one file contains Java methods
- the other file contains the corresponding natural language summaries



In [22]:
with open(train_code_path, "w") as fc, open(train_summary_path, "w") as fs:
    for code, summary in train_pairs:
        fc.write(code + "\n")
        fs.write(summary + "\n")

with open(val_code_path, "w") as fc, open(val_summary_path, "w") as fs:
    for code, summary in val_pairs:
        fc.write(code + "\n")
        fs.write(summary + "\n")

print("Dataset files written successfully.")

Dataset files written successfully.


### Verifying the Embedding Pipeline with Sample Data

Running the script on the sample dataset serves as a sanity check to confirm that the embedding generation process works before applying it to the full dataset of extracted Java methods and summaries.

The assignment provides the script:
```
get_codet5_embeddings.py
```
which tokenizes code or summary text and extracts **pretrained CodeT5 embeddings**.

Running the script on a small sample file allows us to confirm that:

- the script is accessible
- dependencies are installed correctly
- the embedding output format is valid
---
### Output File

The script produces a .pt file:
```
sample_code.pt
```

In [23]:
!python {SHARED_DIR}get_codet5_embeddings.py \
--input {SHARED_DIR}sample_code.txt \
--output {WORK_DIR}sample_code.pt

Loading tokenizer and model: Salesforce/codet5p-220m
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
tokenizer_config.json: 1.48kB [00:00, 6.43MB/s]
vocab.json: 703kB [00:00, 22.7MB/s]
merges.txt: 294kB [00:00, 54.2MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 13.4kB/s]
special_tokens_map.json: 12.5kB [00:00, 33.4MB/s]
config.json: 100% 768/768 [00:00<00:00, 5.66MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
pytorch_model.bin: 100% 446M/446M [00:08<00:00, 51.2MB/s]
Model loaded.
Embedding matrix shape: torch.S

### Generating Embeddings for Sample Summaries

After verifying the embedding pipeline with sample code inputs, the next step is to test the embedding generation process on **natural language summaries**.

The provided script `get_codet5_embeddings.py` is executed on the sample summary dataset
---
### Output File

The script produces the file:
```
sample_summary.pt
```

In [24]:
!python {SHARED_DIR}get_codet5_embeddings.py \
--input {SHARED_DIR}sample_summary.txt \
--output {WORK_DIR}sample_summary.pt \
--max_length 128

Loading tokenizer and model: Salesforce/codet5p-220m
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Model loaded.
Embedding matrix shape: torch.Size([32100, 768])
  Vocab size:     32100
  Embedding dim:  768
Loaded 5 samples from /content/drive/MyDrive/Assignment-2/sample_summary.txt
Tokenizing: 100% 5/5 [00:00<00:00, 1959.96it/s]

Token length stats:
  Mean: 11.2
  Max:  18
  Min:  6

Saved to /content/drive/MyDrive/genai_assignment2_alice/sample_summary.pt


### Generating Embeddings for the Training Dataset

After preparing the training dataset files (`train_code.txt` and `train_summary.txt`), the next step is to convert them into tokenized representations and extract pretrained embeddings.

Using pretrained embeddings allows the LSTM model to start with vector representations that already capture semantic information about both code and natural language.

---

### Generating Code Embeddings

The Java methods in the training dataset are processed using the embedding script:

```
!python {SHARED_DIR}get_codet5_embeddings.py \
--input {WORK_DIR}train_code.txt \
--output {WORK_DIR}train_code.pt
```
The same process is applied to the natural language summaries.

---
### Output Files

This step generates two key files in the working directory:
```
train_code.pt
train_summary.pt
```


In [25]:
!python {SHARED_DIR}get_codet5_embeddings.py \
--input {WORK_DIR}train_code.txt \
--output {WORK_DIR}train_code.pt

Loading tokenizer and model: Salesforce/codet5p-220m
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Model loaded.
Embedding matrix shape: torch.Size([32100, 768])
  Vocab size:     32100
  Embedding dim:  768
Loaded 49000 samples from /content/drive/MyDrive/genai_assignment2_alice/train_code.txt
Tokenizing: 100% 49000/49000 [00:15<00:00, 3162.21it/s]

Token length stats:
  Mean: 60.1
  Max:  512
  Min:  6

Saved to /content/drive/MyDrive/genai_assignment2_alice/train_code.pt


In [26]:
!python {SHARED_DIR}get_codet5_embeddings.py \
--input {WORK_DIR}train_summary.txt \
--output {WORK_DIR}train_summary.pt \
--max_length 128

Loading tokenizer and model: Salesforce/codet5p-220m
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Model loaded.
Embedding matrix shape: torch.Size([32100, 768])
  Vocab size:     32100
  Embedding dim:  768
Loaded 49000 samples from /content/drive/MyDrive/genai_assignment2_alice/train_summary.txt
Tokenizing: 100% 49000/49000 [00:04<00:00, 10596.38it/s]

Token length stats:
  Mean: 16.0
  Max:  87
  Min:  3

Saved to /content/drive/MyDrive/genai_assignment2_alice/train_summar

### Generating Embeddings for the Validation Dataset

In addition to the training dataset, embeddings must also be generated for the **validation dataset**. The validation set is used during training to evaluate model performance and implement **early stopping based on BLEU-1**.

The same embedding generation process used for the training data is applied to the validation data.

---

### Output
```
val_code.pt
```

In [27]:
!python {SHARED_DIR}get_codet5_embeddings.py \
--input {WORK_DIR}val_code.txt \
--output {WORK_DIR}val_code.pt

!python {SHARED_DIR}get_codet5_embeddings.py \
--input {WORK_DIR}val_summary.txt \
--output {WORK_DIR}val_summary.pt \
--max_length 128

Loading tokenizer and model: Salesforce/codet5p-220m
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Model loaded.
Embedding matrix shape: torch.Size([32100, 768])
  Vocab size:     32100
  Embedding dim:  768
Loaded 1000 samples from /content/drive/MyDrive/genai_assignment2_alice/val_code.txt
Tokenizing: 100% 1000/1000 [00:00<00:00, 6120.85it/s]

Token length stats:
  Mean: 39.5
  Max:  406
  Min:  8

Saved to /content/drive/MyDrive/genai_assignment2_alice/val_code.pt
Loading 

## Loading Tokenized Datasets and Embedding Parameters

After generating the tokenized datasets and pretrained embeddings using `get_codet5_embeddings.py`, the resulting `.pt` files are loaded into the notebook.

These files contain the tokenized representation of the dataset as well as the pretrained embedding matrix derived from CodeT5.

This matrix contains pretrained vector representations for the entire vocabulary. Instead of training embeddings from scratch, the LSTM model will reuse these vectors to initialize its embedding layer.



In [28]:
import torch

train_code = torch.load(WORK_DIR + "train_code.pt")
train_summary = torch.load(WORK_DIR + "train_summary.pt")

val_code = torch.load(WORK_DIR + "val_code.pt")
val_summary = torch.load(WORK_DIR + "val_summary.pt")

embedding_matrix = train_code["embedding_matrix"]

pad_id = train_code["pad_token_id"]
eos_id = train_code["eos_token_id"]

vocab_size = train_code["vocab_size"]
embedding_dim = train_code["embedding_dim"]

print("vocab size:", vocab_size)
print("embedding dim:", embedding_dim)

vocab size: 32100
embedding dim: 768


## Creating the PyTorch Dataset

After loading the tokenized datasets and embedding parameters, the next step is to prepare the data so it can be used by the PyTorch training pipeline.

To accomplish this, a custom dataset class called `CodeSummaryDataset` is implemented using PyTorch's `Dataset` interface.


In [29]:
from torch.utils.data import Dataset

class CodeSummaryDataset(Dataset):

    def __init__(self, code_ids, summary_ids):
        self.code_ids = code_ids
        self.summary_ids = summary_ids

    def __len__(self):
        return len(self.code_ids)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.code_ids[idx]),
            torch.tensor(self.summary_ids[idx])
        )

### Create Dataset Objects
These objects represent the **training dataset** and **validation dataset**, which will be used by the model during training.


In [30]:
train_dataset = CodeSummaryDataset(
    train_code["token_ids"],
    train_summary["token_ids"]
)

val_dataset = CodeSummaryDataset(
    val_code["token_ids"],
    val_summary["token_ids"]
)

Pad sequences in each batch so they all have equal length

In [31]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    code, summary = zip(*batch)

    code = pad_sequence(code, batch_first=True, padding_value=pad_id)
    summary = pad_sequence(summary, batch_first=True, padding_value=pad_id)

    return code, summary

## Creating DataLoaders for Training and Validation

DataLoaders provide an efficient way to iterate through the dataset during training and evaluation.


In [32]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    collate_fn=collate_fn
)

## LSTM Encoder–Decoder Model Architecture

This section defines the neural model used to generate natural language summaries from Java methods.

The model follows a **sequence-to-sequence (Seq2Seq) architecture** consisting of:

1. An **embedding layer**
2. An **LSTM encoder**
3. An **LSTM decoder**
4. A **linear output layer**

This architecture allows the model to transform an input sequence of code tokens into a sequence of summary tokens.

---

### Model Definition

The model is implemented as a PyTorch module:
```
class CodeSummaryModel(nn.Module):
```

The constructor initializes the embedding layer, encoder, decoder, and output projection.



In [33]:
import torch.nn as nn

class CodeSummaryModel(nn.Module):

    def __init__(self, embedding_matrix, hidden_size=512):

        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=False
        )

        self.encoder = nn.LSTM(
            embedding_dim,
            hidden_size,
            batch_first=True
        )

        self.decoder = nn.LSTM(
            embedding_dim,
            hidden_size,
            batch_first=True
        )

        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, code, summary):

        code_emb = self.embedding(code)

        _, (h, c) = self.encoder(code_emb)

        summary_emb = self.embedding(summary)

        out, _ = self.decoder(summary_emb, (h, c))

        logits = self.output(out)

        return logits

## Model Initialization and Training Configuration

Initialize the model and configure the components required for training.

This includes:

- selecting the computation device
- initializing the model
- defining the loss function
- configuring the optimizer

---

### Selecting the Computation Device

The notebook automatically selects whether to run on **GPU or CPU**.


In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = CodeSummaryModel(embedding_matrix)

model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

cpu


Install BLEU

In [35]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.3 MB/s eta 0:00:00


## Computing BLEU-1 for Validation

To evaluate the quality of generated summaries during training, the model is periodically evaluated on the **validation dataset** using the BLEU metric.

BLEU  measures the overlap between **generated text** and **reference text** using n-gram matching.
**BLEU-1** is used for early stopping.

---

### Generating Validation Predictions

The function `compute_bleu1()` generates summaries for every example in the validation dataset.


In [36]:
import sacrebleu

def compute_bleu1():

    preds = []
    refs = []

    for code, summary in val_dataset:

        pred_tokens = generate_summary(code, beam_size=3)

        pred_text = tokenizer.decode(pred_tokens, skip_special_tokens=True)
        ref_text = tokenizer.decode(summary.tolist(), skip_special_tokens=True)

        preds.append(pred_text)
        refs.append(ref_text)

    bleu = sacrebleu.corpus_bleu(preds, [refs])

    return bleu.precisions[0]  # BLEU-1

Define early stopping variables

In [37]:
best_bleu = 0
patience = 3
patience_counter = 0

## Training the LSTM Code Summarization Model

Implements the training loop for the encoder–decoder LSTM model. The goal of training is to minimize the difference between the model's predicted summary tokens and the ground-truth summary tokens.

Training is performed using **mini-batch gradient descent**, the **Adam optimizer**, and **cross-entropy loss**.

BLEU-1 measures unigram overlap between generated summaries and reference summaries. This metric is used to monitor model performance on unseen data.

---

## Model Checkpointing

If the validation BLEU-1 score improves, the current model is saved.
## Early Stopping

To prevent overfitting, the training loop implements **early stopping**.

If the validation BLEU-1 score does not improve for several epochs:
```
patience_counter += 1
```

training is stopped:
```
if patience_counter >= patience:
break
```
---
## Summary

This training procedure combines:

- **teacher forcing** for stable sequence learning
- **cross-entropy loss** for token prediction
- **Adam optimization** for efficient parameter updates
- **validation BLEU-1 monitoring**
- **model checkpointing**
- **early stopping**


In [38]:
for code, summary in train_loader:
    print(code.shape)
    print(summary.shape)
    break

torch.Size([32, 508])
torch.Size([32, 31])


In [39]:
CHECKPOINT_PATH = WORK_DIR + "lstm_checkpoint.pt"

In [40]:
best_loss = float("inf")

for epoch in range(10):

    model.train()
    total_loss = 0

    for i, (code, summary) in enumerate(train_loader):

        code = code.to(device)
        summary = summary.to(device)

        optimizer.zero_grad()

        output = model(code, summary[:, :-1])

        loss = criterion(
            output.reshape(-1, vocab_size),
            summary[:, 1:].reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 100 == 0:
            print(f"Epoch {epoch} Batch {i} Loss {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    print("Epoch", epoch, "Avg Loss:", avg_loss)

    # VALIDATION BLEU-1
    bleu1 = compute_bleu1()
    print("Validation BLEU-1:", bleu1)

    if bleu1 > best_bleu:

        best_bleu = bleu1
        patience_counter = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "bleu": bleu1
        }, CHECKPOINT_PATH)

        print("New best checkpoint saved!")

    else:

        patience_counter += 1
        print("No improvement. Patience:", patience_counter)

    if patience_counter >= patience:

        print("Early stopping triggered")
        break

Epoch 0 Batch 0 Loss 10.3795


KeyboardInterrupt: 

## Loading the Best Model Checkpoint

After training completes, the best-performing model checkpoint is loaded for evaluation and summary generation.



In [41]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

model.eval()

CodeSummaryModel(
  (embedding): Embedding(32100, 768)
  (encoder): LSTM(768, 512, batch_first=True)
  (decoder): LSTM(768, 512, batch_first=True)
  (output): Linear(in_features=512, out_features=32100, bias=True)
)

### Load instructor provided test set

In [42]:
import pandas as pd
import ast
import torch

TEST_PATH = SHARED_DIR + "dataset/test_dataset_tokenized.csv"

df_test = pd.read_csv(TEST_PATH)

# convert string lists to python lists
df_test["code_ids"] = df_test["code_ids"].apply(ast.literal_eval)
df_test["summary_ids"] = df_test["summary_ids"].apply(ast.literal_eval)

test_code_ids = df_test["code_ids"].tolist()
test_summary_ids = df_test["summary_ids"].tolist()

print("Test samples:", len(test_code_ids))

Test samples: 99


In [43]:
class TestDataset(torch.utils.data.Dataset):

    def __init__(self, code_ids, summary_ids):
        self.code_ids = code_ids
        self.summary_ids = summary_ids

    def __len__(self):
        return len(self.code_ids)

    def __getitem__(self, idx):
        code = torch.tensor(self.code_ids[idx])
        summary = torch.tensor(self.summary_ids[idx])
        return code, summary


test_dataset = TestDataset(test_code_ids, test_summary_ids)

print("Loaded test dataset:", len(test_dataset))

Loaded test dataset: 99


## Summary Generation Using Beam Search

After training the LSTM encoder–decoder model, summaries are generated for unseen Java methods. This is done using a **beam search decoding algorithm**, which produces higher-quality summaries than simple greedy decoding.

The function `generate_summary()` takes tokenized Java code as input and returns a predicted summary sequence.

Key parameters:

- **beam_size** → number of candidate sequences maintained during decoding  
- **max_len** → maximum length of the generated summary  


The resulting token sequence represents the model's predicted summary.

---


Beam search improves generation quality compared to greedy decoding.

However, this may miss better sequences that require temporarily choosing lower-probability tokens.

Beam search maintains multiple candidate sequences simultaneously, allowing the algorithm to explore more possible summaries before selecting the best one.

Using **beam_size = 3** provides a good balance between computational efficiency and summary quality.


In [44]:
def generate_summary(code_tokens, beam_size=3, max_len=50):

    model.eval()

    code_tokens = code_tokens.unsqueeze(0).to(device)

    with torch.no_grad():

        code_emb = model.embedding(code_tokens)
        _, (h, c) = model.encoder(code_emb)

        beams = [([eos_id], 0.0, h, c)]

        for _ in range(max_len):

            new_beams = []

            for tokens, score, h_i, c_i in beams:

                input_token = torch.tensor([[tokens[-1]]]).to(device)

                emb = model.embedding(input_token)

                out, (h_new, c_new) = model.decoder(emb, (h_i, c_i))

                logits = model.output(out[:, -1, :])

                log_probs = torch.log_softmax(logits, dim=-1)

                topk_probs, topk_tokens = torch.topk(log_probs, beam_size)

                for k in range(beam_size):

                    next_token = topk_tokens[0, k].item()
                    next_score = score + topk_probs[0, k].item()

                    new_beams.append(
                        (tokens + [next_token], next_score, h_new, c_new)
                    )

            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        best_tokens = beams[0][0]

    return best_tokens[1:]

## Generating Summaries for Evaluation

After loading the trained model and implementing the beam search decoding function, the next step is to generate summaries for the test dataset. These predictions will later be compared with the ground-truth summaries to compute evaluation metrics.



In [45]:
from tqdm import tqdm

predictions = []
references = []

with torch.no_grad():
    for code, summary in tqdm(test_dataset):

        pred = generate_summary(code, beam_size=3)

        predictions.append(pred)
        references.append(summary)

100%|██████████| 99/99 [02:56<00:00,  1.78s/it]


Load tokenizers

In [46]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Decoding Token Sequences into Text

The predictions and reference summaries generated in the previous step are currently stored as **token ID sequences**. However, evaluation metrics such as BLEU, METEOR, BERTScore, and SIDE require **natural language text** as input.

Therefore, the token sequences must be converted back into readable text.



In [47]:
decoded_predictions = []
decoded_references = []

for pred, ref in zip(predictions, references):

    pred_text = tokenizer.decode(pred, skip_special_tokens=True)
    ref_text = tokenizer.decode(ref.tolist(), skip_special_tokens=True)

    decoded_predictions.append(pred_text)
    decoded_references.append(ref_text)

## Evaluating Generated Summaries with BLEU


BLEU evaluates the overlap between **generated text** and **reference text** using *n-gram matching*.



In [48]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(decoded_predictions, [decoded_references])

print("BLEU:", bleu.score)
print("BLEU-1:", bleu.precisions[0])
print("BLEU-2:", bleu.precisions[1])
print("BLEU-3:", bleu.precisions[2])
print("BLEU-4:", bleu.precisions[3])

BLEU: 0.28183884328560704
BLEU-1: 13.609467455621301
BLEU-2: 0.9659090909090909
BLEU-3: 0.03006614552014432
BLEU-4: 0.015964240102171137




### Limitations of BLEU for Code Summarization

BLEU measures n-gram overlap and does not account for semantic similarity. Two summaries that convey the same meaning but use different phrasing can receive very low BLEU scores. For this reason, additional metrics such as **METEOR, BERTScore, and SIDE** are also reported to provide a more comprehensive evaluation of the generated summaries.

## Evaluating Generated Summaries with METEOR
METEOR is designed to capture **semantic similarity** between generated text and reference text.



In [49]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [50]:
from nltk.translate.meteor_score import meteor_score

meteor_scores = [
    meteor_score([ref.split()], pred.split())
    for pred, ref in zip(decoded_predictions, decoded_references)
]

print("METEOR:", sum(meteor_scores) / len(meteor_scores))

METEOR: 0.07021359899070292




METEOR scores range from **0 to 1**, where:

- **0** indicates no similarity between the generated and reference summaries  
- **1** indicates a perfect match  





## Evaluating Generated Summaries with BERTScore

Unlike BLEU and METEOR, which rely on surface-level word matching, BERTScore measures **semantic similarity** using contextual embeddings from pretrained transformer models.

In [51]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00


In [52]:
from bert_score import score

P, R, F1 = score(decoded_predictions, decoded_references, lang="en")

print("BERTScore F1:", F1.mean().item())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1: 0.8338578343391418


## SIDE metric

SIDE tries to measure how well the summary aligns with the code itself, not just the reference text.

## Saving Evaluation Data for External Metrics (SIDE)

To compute the **SIDE metric**, the generated summaries, reference summaries, and corresponding code snippets must be saved to text files. These files will later be used as input to the SIDE evaluation repository.


In [53]:
# save predictions
with open("predictions.txt", "w") as f:
    for p in decoded_predictions:
        f.write(p + "\n")

# save references
with open("references.txt", "w") as f:
    for r in decoded_references:
        f.write(r + "\n")

# save code
with open("code.txt", "w") as f:
    for code, _ in test_dataset:
        code_text = tokenizer.decode(code.tolist(), skip_special_tokens=True)
        f.write(code_text + "\n")

clone the SIDE repository

In [54]:
!git clone https://github.com/antonio-mastropaolo/code-summarization-metric.git

Cloning into 'code-summarization-metric'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 94 (delta 36), reused 66 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 4.07 MiB | 11.30 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [55]:
%cd code-summarization-metric
!ls

/content/code-summarization-metric
ablation-st-results.png  LICENSE	   Results     train-hard-negatives.py
Analysis		 README.md	   runEval.py  triplet-example.png
computeAllMetrics.py	 requirements.txt  runTest.py


run SIDE evaluation

In [56]:
from sentence_transformers import SentenceTransformer, util
import torch

In [57]:
model = SentenceTransformer("microsoft/codebert-base")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [58]:
codes = []
preds = []

with open("../code.txt") as f:
    codes = [line.strip() for line in f]

with open("../predictions.txt") as f:
    preds = [line.strip() for line in f]

code_embeddings = model.encode(codes, convert_to_tensor=True)
pred_embeddings = model.encode(preds, convert_to_tensor=True)

scores = util.cos_sim(code_embeddings, pred_embeddings).diagonal()

side_score = scores.mean().item()

print("SIDE score:", side_score)

SIDE score: 0.8340109586715698



The SIDE score measures the **semantic similarity between the generated summary and the source code**, rather than comparing the summary only to a reference summary. This is important for code summarization because the goal of the model is to produce descriptions that accurately reflect the **functionality of the code implementation**.

SIDE uses **CodeBERT embeddings** to represent both code and text in the same semantic space and computes **cosine similarity** between them.

Scores range from **0 to 1**, where:

- **0** indicates no semantic relationship between code and summary  
- **1** indicates perfect semantic alignment  



### Summary

In [59]:
print("BLEU-1:", bleu.precisions[0])
print("BLEU-2:", bleu.precisions[1])
print("BLEU-3:", bleu.precisions[2])
print("BLEU-4:", bleu.precisions[3])
print("METEOR:", sum(meteor_scores)/len(meteor_scores))
print("BERTScore:", F1.mean().item())
print("SIDE:", side_score)

BLEU-1: 13.609467455621301
BLEU-2: 0.9659090909090909
BLEU-3: 0.03006614552014432
BLEU-4: 0.015964240102171137
METEOR: 0.07021359899070292
BERTScore: 0.8338578343391418
SIDE: 0.8340109586715698
